# 📊 Model Evaluation - Challenger Assessment

## Purpose
Runs comprehensive evaluation of the Challenger model using MLflow evaluation framework

## Steps
1. Load Challenger model from Unity Catalog
2. Load validation dataset 
3. Run MLflow model evaluation
4. Generate evaluation report and metrics
5. Log evaluation results for approval workflow

## Notes
- Part of 3-step deployment workflow (Evaluation → Approval → Deployment)
- Follows Iris MLOps pattern for model validation
- Blocks deployment if evaluation fails

In [ ]:
# 📦 Install required packages
%pip install mlflow scikit-learn pandas numpy matplotlib seaborn

# Restart Python 
dbutils.library.restartPython()

In [ ]:
# 📚 Import Libraries
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.models import MetricThreshold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from pyspark.sql import SparkSession
import warnings
warnings.filterwarnings('ignore')

# Initialize
spark = SparkSession.builder.getOrCreate()
mlflow.set_registry_uri("databricks-uc")

print("✅ Libraries loaded for model evaluation")

In [ ]:
# 🎯 Configuration - Get from Bundle Parameters
catalog_name = spark.conf.get("catalog_name", "dev_digital_engineering_services")
schema_name = spark.conf.get("schema_name", "default")
model_name = spark.conf.get("model_name", "water_quality_model")

# Construct full names
FULL_MODEL_NAME = f"{catalog_name}.{schema_name}.{model_name}"
TABLE_NAME = f"{catalog_name}.{schema_name}.water_quality_data"

print(f"🎯 Evaluation Configuration:")
print(f"   📊 Data Table: {TABLE_NAME}")
print(f"   🤖 Model: {FULL_MODEL_NAME}")
print(f"   🎯 Target: Challenger model")

In [ ]:
# 🔍 Load Challenger Model
print("🔍 Loading Challenger model from Unity Catalog...")

try:
    # Load model using Challenger alias
    model_uri = f"models:/{FULL_MODEL_NAME}@Challenger"
    challenger_model = mlflow.sklearn.load_model(model_uri)
    
    print(f"✅ Challenger model loaded successfully")
    print(f"🔍 Model URI: {model_uri}")
    
    # Get model version info
    client = mlflow.tracking.MlflowClient()
    model_version = client.get_model_version_by_alias(FULL_MODEL_NAME, "Challenger")
    
    print(f"📦 Version: {model_version.version}")
    print(f"🏷️ Alias: Challenger")
    
except Exception as e:
    print(f"❌ Error loading Challenger model: {e}")
    print("💡 Make sure training job completed and registered Challenger model")
    raise e

In [ ]:
# 📊 Load Evaluation Dataset
print("📊 Loading evaluation dataset...")

# Load data from Unity Catalog
water_spark_df = spark.sql(f"SELECT * FROM {TABLE_NAME}")
water_df = water_spark_df.toPandas()

# Feature engineering (MUST match training)
feature_cols = ['ph', 'Hardness', 'Solids', 'Chloramines', 'Sulfate', 
               'Conductivity', 'Organic_carbon', 'Trihalomethanes', 'Turbidity']

water_df['ph_normalized'] = water_df['ph'] / 14
water_df['hardness_ratio'] = water_df['Hardness'] / (water_df['Solids'] + 1)
water_df['organic_load'] = water_df['Organic_carbon'] * water_df['Chloramines']
water_df['water_quality_index'] = (
    water_df['ph_normalized'] + 
    (1 / (water_df['Turbidity'] + 1)) + 
    (1 / (water_df['hardness_ratio'] + 1))
) / 3

final_features = feature_cols + ['ph_normalized', 'hardness_ratio', 'organic_load', 'water_quality_index']

# Split for evaluation (use 20% as held-out evaluation set)
eval_data = water_df.sample(frac=0.2, random_state=99)  # Different seed than training
X_eval = eval_data[final_features]
y_eval = eval_data['Potability']

print(f"✅ Evaluation dataset prepared:")
print(f"   Samples: {len(X_eval)}")
print(f"   Features: {len(final_features)}")
print(f"   Target distribution: {y_eval.value_counts().to_dict()}")

In [ ]:
# 🔬 Run MLflow Model Evaluation
print("🔬 Running comprehensive model evaluation...")

with mlflow.start_run(run_name="challenger_evaluation") as run:
    
    # Define evaluation thresholds (water quality specific)
    thresholds = {
        "accuracy_score": MetricThreshold(
            threshold=0.75,  # Minimum 75% accuracy for water safety
            min_absolute_change=0.05,
            min_relative_change=0.05,
            greater_is_better=True
        ),
        "precision_score": MetricThreshold(
            threshold=0.70,  # High precision important for safety
            greater_is_better=True
        ),
        "recall_score": MetricThreshold(
            threshold=0.70,  # High recall important to catch unsafe water
            greater_is_better=True
        ),
        "f1_score": MetricThreshold(
            threshold=0.70,
            greater_is_better=True
        )
    }
    
    # Run MLflow evaluation
    eval_result = mlflow.evaluate(
        model=model_uri,
        data=X_eval,
        targets=y_eval,
        model_type="classifier",
        evaluators=["default"],
        validation_thresholds=thresholds,
        custom_metrics=None,
        extra_metrics=None,
        custom_artifacts=None
    )
    
    # Log evaluation metadata
    mlflow.log_param("eval_samples", len(X_eval))
    mlflow.log_param("eval_features", len(final_features))
    mlflow.log_param("model_alias", "Challenger")
    mlflow.log_param("model_version", model_version.version)
    
    print(f"✅ MLflow evaluation completed!")
    
    eval_run_id = run.info.run_id

In [ ]:
# 📊 Custom Evaluation Metrics for Water Quality
print("📊 Computing custom water quality metrics...")

# Get predictions
y_pred = challenger_model.predict(X_eval)
y_pred_proba = challenger_model.predict_proba(X_eval)

# Calculate additional metrics relevant to water quality
custom_metrics = {}

# Safety-critical metrics
# False Negative Rate (missing unsafe water - very dangerous!)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_eval, y_pred)
fn = cm[1, 0]  # False negatives (unsafe water predicted as safe)
tp = cm[1, 1]  # True positives
fnr = fn / (fn + tp) if (fn + tp) > 0 else 0

# False Positive Rate (safe water predicted as unsafe - economic cost)
fp = cm[0, 1]  # False positives 
tn = cm[0, 0]  # True negatives
fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

custom_metrics['false_negative_rate'] = fnr  # Critical for safety
custom_metrics['false_positive_rate'] = fpr  # Economic impact
custom_metrics['safety_score'] = 1 - fnr     # Higher is safer

# Log custom metrics
with mlflow.start_run(run_id=eval_run_id):
    for metric_name, value in custom_metrics.items():
        mlflow.log_metric(f"custom_{metric_name}", value)

print(f"🔍 Water Quality Safety Metrics:")
print(f"   False Negative Rate: {fnr:.4f} (Critical - unsafe water missed)")
print(f"   False Positive Rate: {fpr:.4f} (Economic - safe water rejected)")
print(f"   Safety Score: {custom_metrics['safety_score']:.4f}")

In [ ]:
# ✅ Evaluation Results Summary
print("✅ EVALUATION SUMMARY:")
print("=" * 50)

# Check if evaluation passed thresholds
passed_evaluation = True
failed_metrics = []

# Check each threshold
basic_metrics = {
    'accuracy': accuracy_score(y_eval, y_pred),
    'precision': precision_score(y_eval, y_pred, average='weighted'),
    'recall': recall_score(y_eval, y_pred, average='weighted'),
    'f1': f1_score(y_eval, y_pred, average='weighted')
}

threshold_values = {
    'accuracy': 0.75,
    'precision': 0.70,
    'recall': 0.70,
    'f1': 0.70
}

for metric, value in basic_metrics.items():
    threshold = threshold_values[metric]
    status = "✅ PASS" if value >= threshold else "❌ FAIL"
    print(f"{metric.capitalize():12}: {value:.4f} (threshold: {threshold:.2f}) {status}")
    
    if value < threshold:
        passed_evaluation = False
        failed_metrics.append(metric)

# Safety check
safety_threshold = 0.90  # Very high safety standard
safety_pass = custom_metrics['safety_score'] >= safety_threshold
safety_status = "✅ PASS" if safety_pass else "❌ FAIL"
print(f"Safety Score : {custom_metrics['safety_score']:.4f} (threshold: {safety_threshold:.2f}) {safety_status}")

if not safety_pass:
    passed_evaluation = False
    failed_metrics.append("safety_score")

# Final evaluation decision
print(f"\n🎯 EVALUATION DECISION:")
if passed_evaluation:
    print("✅ CHALLENGER MODEL PASSED EVALUATION")
    print("🎯 Ready for approval workflow")
    
    # Tag the model version as evaluated
    client.set_model_version_tag(
        name=FULL_MODEL_NAME,
        version=model_version.version,
        key="evaluation_status",
        value="passed"
    )
else:
    print(f"❌ CHALLENGER MODEL FAILED EVALUATION")
    print(f"📊 Failed metrics: {failed_metrics}")
    print("🔄 Consider retraining with different parameters")
    
    # Tag the model version as failed
    client.set_model_version_tag(
        name=FULL_MODEL_NAME,
        version=model_version.version,
        key="evaluation_status",
        value="failed"
    )
    
    # Raise error to stop deployment pipeline
    raise Exception(f"Model evaluation failed on metrics: {failed_metrics}")

print(f"\n🎯 Next: Run approval workflow to check human approval")